In [1]:
import torch
import torch.nn.functional as F
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


SIGN_BIT = 1
GREATER_THAN_ONE_BIT = 1


@dataclass
class MXTensorConfig:
    mantissa_bits: int
    exponent_bits: int
    overflow_mask: int
    
    
@dataclass
class MXTensor:
    shared_exp: torch.Tensor   # [batch, num_blocks]
    data: torch.Tensor         # [batch, num_blocks, block_size]
    original_shape: tuple
    exponent_bits: torch.Tensor
    mantissa_bits: torch.Tensor
    e_min_layer: torch.Tensor

mx_config_factory = dict()
mx_config_factory['mx-e4bm8'] = MXTensorConfig(mantissa_bits=torch.tensor(8), exponent_bits=torch.tensor(4), overflow_mask=None)

    
def mx_quantizer(x: torch.Tensor, block_size: int = 32, mx_type : str = "mx-e4bm8", skip_exp : bool = False, calibrate_offset : int = 0) -> MXTensor:
    # 1. Pad to a multiple of block_size
    width = x.shape[-1]
    original_shape = tuple(x.shape)
    num_blocks = math.ceil(width / block_size)

    width = x.shape[-1]
    num_blocks = math.ceil(width / block_size)
    pad_amount = num_blocks * block_size - width
    
    if pad_amount > 0:
        x_pad = F.pad(x, (0, pad_amount), mode='constant', value=0)
    else:
        x_pad = x

    # Now x_flat is [B*C*H, num_blocks * block_size]
    # We reshape to [B*C*H, num_blocks, block_size]
    blocks = x_pad.view([*x_pad.shape[:-1], num_blocks, block_size])

    # 2. Per-block floor(log2(|x|)), 0 for all-zero blocks (avoids -inf -> NaN)
    abs_blocks = blocks.abs()
    mask = abs_blocks > 0
    log2_vals = torch.zeros_like(abs_blocks)
    log2_vals[mask] = torch.log2(abs_blocks[mask]).floor()
    log2_scale = log2_vals.amax(dim=-1)                                  # [batch, num_blocks]

    # 3. Shared exponent / shift amount
    e_min_layer = mx_config_factory[mx_type].mantissa_bits - SIGN_BIT - GREATER_THAN_ONE_BIT + calibrate_offset
    shift = log2_scale - e_min_layer 

    shift_sat = torch.where(shift > -e_min_layer, shift, -e_min_layer)
    shift_sat = torch.where(shift_sat < torch.pow(2, mx_config_factory[mx_type].exponent_bits), shift_sat, torch.pow(2, mx_config_factory[mx_type].exponent_bits))  # not that since we use exponent bias (e_min_layer) the exponent itself remains unsigned
    
    # 4. Divide each block by its shared scale
    if skip_exp:
        # in case the input is already integer, just add zeros exponent field and don't quantize the number
        shift_sat = torch.zeros_like(shift_sat)
        data = blocks

    divisor = torch.exp2(shift_sat.unsqueeze(-1).float())
    data = torch.floor(blocks / divisor) 

    # saturation (in the case where user uses the skip_exp flag but has more bits than the mantissa has)
    data = torch.where(data > torch.pow(2, mx_config_factory[mx_type].mantissa_bits-1)-1, torch.pow(2, mx_config_factory[mx_type].mantissa_bits-1)-1, data)  # overflow
    data = torch.where(data < -torch.pow(2, mx_config_factory[mx_type].mantissa_bits-1), -torch.pow(2, mx_config_factory[mx_type].mantissa_bits-1), data)  # underflow

    if mx_config_factory[mx_type].overflow_mask:
        raise NotImplementedError("Do not handle overflow masking for now [TBD]")
    
    return MXTensor(
        shared_exp=shift_sat, 
        data=data, 
        original_shape=original_shape, 
        exponent_bits=mx_config_factory[mx_type].exponent_bits, 
        mantissa_bits=mx_config_factory[mx_type].mantissa_bits, 
        e_min_layer=e_min_layer
    )


def mx_dequantizer(mx: MXTensor) -> torch.Tensor:
    # Multiply data back up by 2^shared_exp, broadcast per block
    divisor = torch.exp2(mx.shared_exp.unsqueeze(-1).float())
    deq = mx.data * divisor                                              # [batch, num_blocks, block]

    # Flatten and strip padding
    flat = deq.reshape(mx.original_shape[0], -1)
    return flat[:, :mx.original_shape[-1]]

# mx conv example

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def align_mx_to_elements(mx: MXTensor):
    """
    Expand shared_exp so every mantissa element has its own exponent,
    then flatten and trim padding.

    Returns:
        data_flat: [..., original_width]
        exp_flat:  [..., original_width]
    """
    num_blocks = mx.data.shape[-2]
    block_size = mx.data.shape[-1]

    data_flat = mx.data.view(*mx.data.shape[:-2], num_blocks * block_size)
    exp_flat = (
        mx.shared_exp
        .unsqueeze(-1)
        .expand_as(mx.data)
        .contiguous()
        .view(*mx.data.shape[:-2], num_blocks * block_size)
    )

    return data_flat[..., :mx.original_shape[-1]], exp_flat[..., :mx.original_shape[-1]]

def tru(x_data_flat: torch.Tensor, 
          x_exp_flat: torch.Tensor, 
          line: int, 
          S_h: int, 
          S_w: int, 
          K_w: int, 
          K_h: int, 
          out_width: int, 
          batch: int):

    h_start = line * S_h
    strip_width = K_w + (out_width - 1) * S_w

    strip = x_data_flat[
        batch, :, h_start:h_start + K_h, :strip_width
    ]
    strip_exp = x_exp_flat[
        batch, :, h_start:h_start + K_h, :strip_width
    ]

    patches = strip.unfold(-1, K_w, S_w)
    patches_exp = strip_exp.unfold(-1, K_w, S_w)

    patches_flat = patches.permute(2, 0, 1, 3).reshape(out_width, -1)
    patches_exp_flat = patches_exp.permute(2, 0, 1, 3).reshape(out_width, -1)

    return patches_flat, patches_exp_flat 




def bcu(
    x_data: torch.Tensor,
    x_exp: torch.Tensor,
    w_data: torch.Tensor,
    w_exp: torch.Tensor,
    acc_bit_width: int = 46
) -> torch.Tensor:
    """
    Hardware-style dot product over the last dimension.
    For each element: multiply mantissas, shift by combined exponent, accumulate.
    """
    prod_mant = x_data * w_data
    combined_exp = x_exp + w_exp
    shifted = (prod_mant * torch.exp2(combined_exp.float())).floor()  # mimic shift right
    p_sum = torch.zeros(x_data.shape[-2])

    # each line in a different cycle
    for k in range(x_data.shape[-2]):
        for l in range(x_data.shape[-1]):
            # Sum element by element to detect overflow
            p_sum[k] += torch.where(p_sum[k] >= 2.0 ** (acc_bit_width-1) -1 or p_sum[k] <= -2.0 ** (acc_bit_width-1), 0, shifted[k][l])
            # p_sum[k] = torch.where(p_sum[k] <= -2.0 ** (acc_bit_width-1), p_sum[k] <= -2.0 ** (acc_bit_width-1), p_sum[k])  # todo

            # Overflow/underflow code: mimic shift right [2**acc_bit_width, 0] 
            p_sum[k] = torch.where(p_sum[k].abs() < 1, 0, p_sum[k])  # shift right max
            p_sum[k] = torch.where(p_sum[k] >= 2.0 ** (acc_bit_width-1) -1 , 2.0 ** (acc_bit_width-1) - 1, p_sum[k])  # overflow
            p_sum[k] = torch.where(p_sum[k] <= -2.0 ** (acc_bit_width-1), -2.0 ** (acc_bit_width-1), p_sum[k])  # underflow
    
    return p_sum


def pcu_pla(x : torch.Tensor, scale : int = 1, bias : int = 0):
    x_tmp = x * scale
    x_tmp = x_tmp + bias
    return x_tmp


def pcu_div(x : torch.Tensor, shift_right : int = 0, round_type: str = "round"):
    # TODO (not now!): add a task that check illegale right shift  
    x_tmp = (x * 2.0 ** (shift_right)).floor()
    if round_type == "floor":
        x_tmp_r = x_tmp.floor()
    elif round_type == "round":
        x_tmp_r = x_tmp.round()
    elif round_type == "trunc":
        x_tmp_r = x_tmp.trunc()
    else:
        raise KeyError("[ERROR] : unknon type of rounding (posible options: floor | round | trunc)")
    return x_tmp_r


def relux(x : torch.Tensor, min : int, max : int):
    x_tmp = torch.clamp(x, min=min, max=max)
    return x_tmp.int()


@dataclass
class Cstruct:
    bias : torch.Tensor
    scale : torch.Tensor
    shift : torch.Tensor
    round_type : torch.Tensor
    min_val : torch.Tensor
    max_val : torch.Tensor


def pcu(p_sum: torch.Tensor, c_struct: Cstruct):
    pla_psum = pcu_pla(p_sum, scale=c_struct.scale, bias=c_struct.bias)
    round_psum = pcu_div(pla_psum, shift_right=c_struct.shift, round_type=c_struct.round_type)
    o_line = relux(round_psum, min=c_struct.min_val, max=c_struct.max_val)
    return o_line


class MXConv2D_Loops(nn.Module):
    def __init__(
        self, in_channels, out_channels, kernel_size,
        stride=1, padding=0, min_val=-1.0, max_val=1.0
    ):
        super(MXConv2D_Loops, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.stride = (stride, stride) if isinstance(stride, int) else stride
        self.padding = (padding, padding) if isinstance(padding, int) else padding

        self.weight = nn.Parameter(torch.randn(
            out_channels, in_channels, *self.kernel_size
        ))
        self.bias = nn.Parameter(torch.zeros(out_channels))

        with torch.no_grad():
            fan_in = in_channels * self.kernel_size[0] * self.kernel_size[1]
            self.weight.data /= math.sqrt(fan_in)
            self.bias.data.uniform_(min_val, max_val)

        # Pre-quantize weights per output channel using mx_quantizer
        self.w_data_list, self.w_exp_list = [], []
        self.c_struct_arr = {}
        self.shift_bias = 0

    def update_c_struct(self, shift_bias=None):
        # this helps update / correct bias param (may be affected by the e_min_layer / input)
        if shift_bias:
            self.shift_bias = shift_bias

        for oc in range(self.out_channels):
            w_flat = self.weight[oc].data.view(-1)
            mx_w = mx_quantizer(w_flat, block_size=block_size)
            w_data_flat, w_exp_flat = align_mx_to_elements(mx_w)
            self.w_data_list.append(w_data_flat)
            self.w_exp_list.append(w_exp_flat)
            c_strcut_tmp = Cstruct(
                bias=(self.bias[oc] * 2.0 ** self.shift_bias).floor() if self.shift_bias else self.bias[oc],
                scale=1,
                shift=0,
                round_type="round",
                min_val=-2.0 ** (24 - 1),
                max_val=2.0 ** (24 - 1) -1
            )
            self.c_struct_arr[oc] = c_strcut_tmp


    def forward(self, mx: MXTensor):
        batch_size, C_in, input_height, input_width = mx.original_shape
        K_h, K_w = self.kernel_size
        S_h, S_w = self.stride

        if C_in != self.in_channels:
            raise ValueError(
                f"Input has {C_in} channels, layer expects {self.in_channels}"
            )

        if self.padding[0] > 0 or self.padding[1] > 0:
            raise NotImplementedError("Padding not yet supported")

        # Align input mantissas with their per-element exponents
        x_data_flat, x_exp_flat = align_mx_to_elements(mx)

        out_height = (input_height + 2 * self.padding[0] - K_h) // S_h + 1
        out_width = (input_width + 2 * self.padding[1] - K_w) // S_w + 1

        output = torch.zeros(
            batch_size, self.out_channels, out_height, out_width,
            device=mx.data.device
        )

        # Convolution loops
        for b in range(batch_size):
            for oc in range(self.out_channels):
                tru_w = self.w_data_list[oc]
                tru_w_exp = self.w_exp_list[oc]

                for oh in range(out_height):
                    # order the input such that it run conv exactly as the npe
                    tru_a, tru_a_exp = tru(
                            x_data_flat, 
                            x_exp_flat, 
                            oh, 
                            S_h, 
                            S_w, 
                            K_w, 
                            K_h, 
                            out_width, 
                            b
                    )

                    # mimic the mac units computation
                    p_sum = bcu(
                        tru_a,
                        tru_a_exp,
                        tru_w,
                        tru_w_exp
                    )

                    # mimic the post bcu computation
                    o_line = pcu(p_sum, c_struct=self.c_struct_arr[oc])

                    # return a line in an output channel in a batch
                    output[b, oc, oh, :] = o_line

        return output



# --- Testing the implementation ---
if __name__ == "__main__":
    # Hyperparameters
    B, C_in, C_out, H, W = 1, 1, 3, 32, 32
    K, S, P = 3, 1, 0
    block_size = 32
    
    # Input
    x = (torch.rand(B, C_in, H, W) -0.5) * 2

    x_int = (x * 2 ** 6).round() 
    
    # Custom Loop Layer
    custom_conv = MXConv2D_Loops(C_in, C_out, K, S, P)
    
    # PyTorch Standard Layer for comparison
    torch_conv = nn.Conv2d(C_in, C_out, K, stride=S, padding=P)

    # Copy weights for fair comparison
    with torch.no_grad():
        torch_conv.weight.copy_(custom_conv.weight)
        torch_conv.bias.copy_(custom_conv.bias)

    mx = mx_quantizer(x_int, block_size=block_size, calibrate_offset=0)
    custom_conv.update_c_struct(shift_bias=6)
    out_custom = custom_conv(mx) / 2 ** 6

    print(out_custom)
    print("---------------------------------------------------------")
    bits = 7
    x_int_t = (x * 2 ** bits).round() 
    custom_conv2 = MXConv2D_Loops(C_in, C_out, K, S, P)
    # Copy weights for fair comparison
    with torch.no_grad():
        custom_conv2.weight.copy_(custom_conv.weight)
        custom_conv2.bias.copy_(custom_conv.bias)


    mx = mx_quantizer(x_int_t, block_size=block_size, skip_exp=True, calibrate_offset=0)
    custom_conv2.update_c_struct(shift_bias=bits)
    out_custom2 = custom_conv2(mx) / 2 ** bits
    print(out_custom2)
    print("========================================================")
    # print(x)
    # print(x_int)
    out_torch = torch_conv(x)
    print(out_torch)

    # diff
    diff = torch.abs(out_custom - out_torch).max()
    print(f"Max difference between custom [int6] and torch: {diff.item():.6f}")
    diff = torch.abs(out_custom2 - out_torch).max()
    print(f"Max difference between custom [int{bits+1}] and torch: {diff.item():.6f}")

tensor([[[[-0.4062,  0.3281,  0.0469,  ..., -0.1875,  0.3125,  0.0781],
          [-0.0781, -0.0781, -0.2031,  ...,  0.1250,  0.3594, -0.1406],
          [ 0.0312, -0.3438, -0.7031,  ..., -0.0781, -0.1094, -0.3594],
          ...,
          [-0.5000, -0.1719, -0.1875,  ..., -0.1562, -0.0938, -0.3750],
          [-0.7656, -0.0625, -0.5938,  ..., -0.3906, -0.0781, -0.3906],
          [-0.7344,  0.0000, -0.5625,  ..., -0.3594, -0.2031, -0.1875]],

         [[ 0.8281,  0.0000,  0.2188,  ...,  0.2344, -0.0938, -0.2656],
          [ 0.2500, -0.2969,  0.5938,  ...,  0.2812, -0.0469,  0.7500],
          [ 0.3906, -0.2656,  0.7656,  ...,  0.1719,  0.5000,  0.3281],
          ...,
          [ 0.5000,  0.8281, -0.4219,  ...,  0.3281,  1.2031,  0.0469],
          [ 0.0312,  0.5312,  0.7344,  ..., -0.0156,  0.0625,  0.7344],
          [ 0.7188, -0.6719,  0.4219,  ...,  0.8750, -0.3906, -0.4375]],

         [[-1.2656,  0.3750, -0.7031,  ..., -0.2344,  0.3750, -0.9219],
          [-0.5938, -0.1562, -